# 01 Collect Raw PR Suggestion Pairs

Purpose: collect raw suggestion/landed-diff pairs from closed PRs.

This notebook should only create or refresh raw input data. It should not create labels, review files, training rows, or metric reports.

Output:

```text
data/raw/pipeline-fl-control-plane-closed-prs/export/raw_pairs.jsonl
```

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
RAW_EXPORT_DIR = PROJECT_ROOT / "data" / "raw" / "pipeline-fl-control-plane-closed-prs" / "export"
RAW_PAIRS_JSONL = RAW_EXPORT_DIR / "raw_pairs.jsonl"
RAW_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.tools.sap/Lenny/pipeline-fl-control-plane"
LIMIT = 120
GITHUB_CONCURRENCY = 12
COMMENT_SOURCE = "hyperspace"  # use "all" for every suggestion fence source
REVIEW_COMMENTS_ONLY = True

RAW_PAIRS_JSONL

## Load Credentials

The collection script reads `GITHUB_TOOL_TOKEN` for `github.tools.sap`. This cell loads local env files without printing secrets.

In [ ]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("'").strip('"'))

load_env_file(PROJECT_ROOT / ".env")
load_env_file(PROJECT_ROOT / ".env.pr-pairs.local")

if "GITHUB_TOOL_TOKEN" not in os.environ and "GITHUB_TOKEN" in os.environ:
    os.environ["GITHUB_TOOL_TOKEN"] = os.environ["GITHUB_TOKEN"]

missing = [name for name in ["GITHUB_TOOL_TOKEN"] if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Missing env var(s): {missing}. Add them to .env.pr-pairs.local and rerun this cell.")

print("Env OK. Token is loaded but not printed.")

## Collect Raw Pairs

This is the only cell that calls GitHub and refreshes raw data.

In [ ]:
cmd = [
    "uv",
    "run",
    "--locked",
    "--extra",
    "collection",
    "pr-suggestion-collect",
    "--github-closed-prs-repo",
    REPO_URL,
    "--collect-pairs",
    "--suggestions-from-pr-comments",
    "--comment-source",
    COMMENT_SOURCE,
    "--review-comments-only",
    "--github-concurrency",
    str(GITHUB_CONCURRENCY),
    "--limit",
    str(LIMIT),
    "--output",
    str(RAW_PAIRS_JSONL),
]

env = os.environ.copy()
env.setdefault("UV_CACHE_DIR", "/private/tmp/uv-cache")

print("Writing:", RAW_PAIRS_JSONL)
print("Scanning closed PRs:", REPO_URL)
subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)

## Inspect Raw Output

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw_pairs = read_jsonl(RAW_PAIRS_JSONL)
print("pairs:", len(raw_pairs))
print("unique PRs:", len({row["pr_url"] for row in raw_pairs}))
print("top PRs by suggestions:")
for pr_url, count in Counter(row["pr_url"] for row in raw_pairs).most_common(10):
    print(f"{count:>3}  {pr_url}")

[
    {
        "inspection_id": row["inspection_id"],
        "fault_id": row["fault_id"],
        "pr_url": row["pr_url"],
        "handler_diff_path": row["handler_diff_path"],
        "suggestion_chars": len(row["handler_code_changes_diff"]),
        "merged_diff_chars": len(row["merged_pr_diff"]),
    }
    for row in raw_pairs[:20]
]